# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [2]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', QBC12_STUDENT_ID) or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

PROJECT: e:\MLOps\Week_03\01
EXPERIMENT_NAME: qbc12_hw02_student_elaheh_badali


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [3]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', MLFLOW_TRACKING_USERNAME) or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', MLFLOW_TRACKING_PASSWORD) or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/sklearn_pipeline_model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

Best run ID  : 6850fbb8dc784ea0850400865ddae4a1
Model URI    : runs:/6850fbb8dc784ea0850400865ddae4a1/sklearn_pipeline_model
Run name     : v5_clean_random_forest
F1 score     : 0.9881324172392255


In [4]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'model']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [5]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'has_reviews_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 34)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,...,338.0,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0,1
1,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,730,...,338.0,46,0.505495,2.0,730.0,15,0.483871,2.0,730.0,0
2,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,730,...,337.0,43,0.472527,2.0,730.0,17,0.548387,2.0,730.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [6]:
# TODO 1.3
# Write your reproducibility check here.

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

# Your code here
X_sample = X_sample.astype(object)
X_sample = X_sample.where(~X_sample.isna(), np.nan)

preds_1 = model.predict(X_sample)
preds_2 = model.predict(X_sample)

if not np.array_equal(preds_1, preds_2):
    raise ValueError("Reproducibility check failed: predictions are different.")

matches = (preds_1 == y_sample.to_numpy()).sum()
accuracy = matches / len(y_sample)

print("Matching predictions:", matches)
print("Sample accuracy:", accuracy)

print("Reproducibility check passed.")

Matching predictions: 47
Sample accuracy: 0.94
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [8]:
# TODO 2.1
# Create src/airbnb_serving/schema.py
#
# Define two Pydantic models:
#
# ListingFeatures:
#   - all feature columns from FEATURE_COLS above
#   - use correct types: str, int, float, bool
#   - nullable fields (those with NaN in dataset) should use Optional[float] = None
#
# PredictionResponse:
#   - listing_id: int | None = None
#   - prediction: int  (0 or 1)
#   - probability_high_demand: float
#   - model_run_id: str
#
# Write your code here.

(PROJECT / 'src/airbnb_serving/schema.py').write_text(textwrap.dedent("""
from typing import Optional

from pydantic import BaseModel


class ListingFeatures(BaseModel):
    room_type: str
    property_type: str
    neighbourhood_name: str

    accommodates: int
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: Optional[float] = None

    minimum_nights: int
    maximum_nights: int

    instant_bookable: bool
    is_superhost: bool

    host_listing_count: int

    has_reviews_before_cutoff: bool
    total_reviews_before_cutoff: int
    unique_reviewers_before_cutoff: int
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None

    available_days_last_90d: int
    available_rate_last_90d: float
    avg_minimum_nights_calendar_last_90d: float
    avg_maximum_nights_calendar_last_90d: float

    available_days_last_30d: int
    available_rate_last_30d: float
    avg_minimum_nights_calendar_last_30d: float
    avg_maximum_nights_calendar_last_30d: float


class PredictionResponse(BaseModel):
    listing_id: int | None = None
    prediction: int
    probability_high_demand: float
    model_run_id: str
""").strip() + "\n")


1229

### 2.2 Prediction logic

In [9]:
# TODO 2.2
# Create src/airbnb_serving/predictor.py
#
# Add two functions:
#
# predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse
#   - convert ListingFeatures to a single-row DataFrame
#   - call model.predict and model.predict_proba
#   - return PredictionResponse
#   - all values must be native Python types (int, float), not numpy types
#
# predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]
#   - convert list to a multi-row DataFrame in one step
#   - call model.predict and model.predict_proba once for the whole batch
#   - return a list of PredictionResponse
#   - do NOT loop and call predict_single for each row

(PROJECT / 'src/airbnb_serving/predictor.py').write_text(textwrap.dedent('''
import pandas as pd

from airbnb_serving.schema import ListingFeatures, PredictionResponse


def _features_to_dict(features: ListingFeatures) -> dict:
    return features.model_dump()


def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    row = _features_to_dict(features)
    X = pd.DataFrame([row])

    prediction = model.predict(X)[0]
    probability = model.predict_proba(X)[0][1]

    return PredictionResponse(
        listing_id=None,
        prediction=int(prediction),
        probability_high_demand=float(probability),
        model_run_id=str(run_id),
    )


def predict_batch(
    features_list: list[ListingFeatures],
    model,
    run_id: str,
) -> list[PredictionResponse]:
    rows = [_features_to_dict(features) for features in features_list]
    X = pd.DataFrame(rows)

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    responses = []
    for prediction, probability in zip(predictions, probabilities):
        responses.append(
            PredictionResponse(
                listing_id=None,
                prediction=int(prediction),
                probability_high_demand=float(probability),
                model_run_id=str(run_id),
            )
        )

    return responses
''').strip() + '\n')

1287

### 2.3 FastAPI app

In [10]:
# TODO 2.3
# Create src/airbnb_serving/app.py
#
# Build a FastAPI app with:
#
#   GET /health
#     response: {"status": "ok", "model_run_id": str}
#
#   POST /predict
#     request body: ListingFeatures
#     response: PredictionResponse
#
#   POST /predict/batch
#     request body: list[ListingFeatures]
#     response: list[PredictionResponse]
#
# Rules:
#   - Load the model ONCE using a lifespan context manager, not inside handlers
#   - Read MODEL_RUN_ID, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME,
#     MLFLOW_TRACKING_PASSWORD from environment variables
#   - Use the predictor module for prediction logic

(PROJECT / 'src/airbnb_serving/app.py').write_text(textwrap.dedent('''
import os
from contextlib import asynccontextmanager

import mlflow
import mlflow.sklearn
from fastapi import FastAPI

from airbnb_serving.predictor import predict_batch, predict_single
from airbnb_serving.schema import ListingFeatures, PredictionResponse


model = None
model_run_id = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global model, model_run_id

    model_run_id = os.environ["MODEL_RUN_ID"]

    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
    username = os.environ.get("MLFLOW_TRACKING_USERNAME")
    password = os.environ.get("MLFLOW_TRACKING_PASSWORD")

    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    if username:
        os.environ["MLFLOW_TRACKING_USERNAME"] = username

    if password:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = password

    model_uri = f"runs:/{model_run_id}/sklearn_pipeline_model"
    model = mlflow.sklearn.load_model(model_uri)

    yield

    model = None
    model_run_id = None


app = FastAPI(
    title="Airbnb Demand Prediction API",
    version="1.0.0",
    lifespan=lifespan,
)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_run_id": str(model_run_id),
    }


@app.post("/predict", response_model=PredictionResponse)
def predict_endpoint(features: ListingFeatures):
    return predict_single(features, model, str(model_run_id))


@app.post("/predict/batch", response_model=list[PredictionResponse])
def predict_batch_endpoint(features_list: list[ListingFeatures]):
    return predict_batch(features_list, model, str(model_run_id))
''').strip() + '\n')

1586

### 2.4 Package metadata

In [11]:
# TODO 2.4
# Create pyproject.toml and requirements.txt
#
# Package name: airbnb-serving
# Source directory: src/
# Required dependencies:
#   fastapi>=0.111
#   uvicorn[standard]>=0.29
#   mlflow>=2.13
#   scikit-learn>=1.4
#   pandas>=2.0
#   pydantic>=2.0
#   python-dotenv>=1.0

# Write your code here.
(PROJECT / 'pyproject.toml').write_text(textwrap.dedent('''
[project]
name = "airbnb-serving"
version = "0.1.0"
description = "FastAPI service for Airbnb demand prediction"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.111",
    "uvicorn[standard]>=0.29",
    "mlflow>=2.13",
    "scikit-learn>=1.4",
    "pandas>=2.0",
    "pydantic>=2.0",
    "python-dotenv>=1.0",
]

[tool.setuptools.packages.find]
where = ["src"]

[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"
''').strip() + '\n')


(PROJECT / 'requirements.txt').write_text(textwrap.dedent('''
fastapi>=0.111
uvicorn[standard]>=0.29
mlflow>=2.13
scikit-learn>=1.4
pandas>=2.0
pydantic>=2.0
python-dotenv>=1.0
''').strip() + '\n')

115

### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [12]:
import sys
!{sys.executable} -m pip install -q -e .

In [13]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

Server started, PID: 20120


In [14]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'has_reviews_before_cutoff': True,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

Health: {'status': 'ok', 'model_run_id': '6850fbb8dc784ea0850400865ddae4a1'}
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.0714576865441177, 'model_run_id': '6850fbb8dc784ea0850400865ddae4a1'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [15]:
# TODO 2.6
# Write your benchmark here.
# Hint: convert df rows to list of dicts with df.to_dict(orient='records')

import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

# Your benchmark code here

# --- Print results ---
# Example output format:
# Method        | Total (s) | Per prediction (ms)
# single loop   |   X.XX    |       XX.X
# batch         |   X.XX    |        X.X
# Speedup: Xx

# --- Answer ---
# Why is batch faster? (write as a comment below)
# ...


import time
import math
import requests
import numpy as np
import pandas as pd

BASE_URL = "http://localhost:12345"
BENCHMARK_SIZE = 100

def clean_for_json(row: dict) -> dict:
    clean = {}

    for k, v in row.items():
        if pd.isna(v):
            clean[k] = None
        elif isinstance(v, np.integer):
            clean[k] = int(v)
        elif isinstance(v, np.floating):
            clean[k] = float(v)
        elif isinstance(v, np.bool_):
            clean[k] = bool(v)
        else:
            clean[k] = v

    return clean


benchmark_df = df[FEATURE_COLS].dropna().head(BENCHMARK_SIZE)

benchmark_rows = [
    clean_for_json(row)
    for row in benchmark_df.to_dict(orient="records")
]

print("Benchmark rows:", len(benchmark_rows))

for i, row in enumerate(benchmark_rows):
    response = requests.post(f"{BASE_URL}/predict", json=row)
    if response.status_code != 200:
        print("Bad row index:", i)
        print("Status:", response.status_code)
        print(response.text)
        raise ValueError("Invalid benchmark payload")

start = time.perf_counter()

for row in benchmark_rows:
    response = requests.post(f"{BASE_URL}/predict", json=row)
    response.raise_for_status()

single_total = time.perf_counter() - start

start = time.perf_counter()

response = requests.post(
    f"{BASE_URL}/predict/batch",
    json=benchmark_rows,
)
response.raise_for_status()

batch_total = time.perf_counter() - start

actual_size = len(benchmark_rows)

single_per_ms = (single_total / actual_size) * 1000
batch_per_ms = (batch_total / actual_size) * 1000
speedup = single_total / batch_total

print(f"{'Method':<15} | {'Total (s)':<10} | {'Per prediction (ms)'}")
print("-" * 55)
print(f"{'single loop':<15} | {single_total:<10.4f} | {single_per_ms:.2f}")
print(f"{'batch':<15} | {batch_total:<10.4f} | {batch_per_ms:.2f}")
print("-" * 55)
print(f"Speedup: {speedup:.2f}x")

# Why is batch faster?
# Batch is faster because it sends one HTTP request instead of 100 separate requests.
# It also creates one DataFrame and runs the model pipeline once for all rows,
# instead of repeating request parsing, DataFrame creation, preprocessing,
# and model inference 100 separate times.

Benchmark rows: 100
Method          | Total (s)  | Per prediction (ms)
-------------------------------------------------------
single loop     | 220.8395   | 2208.40
batch           | 2.2040     | 22.04
-------------------------------------------------------
Speedup: 100.20x


In [16]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [17]:
# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

# Write your code here.
(PROJECT / 'Dockerfile.naive').write_text(textwrap.dedent('''
FROM python:3.11

WORKDIR /app

COPY . .
                                                          
RUN pip install --upgrade pip
RUN pip install -r requirements.txt 
RUN pip install -e .

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
''').strip() + '\n')

225

### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [18]:
# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

# Write your code here.

(PROJECT / 'Dockerfile').write_text(textwrap.dedent('''
# ---------- Builder Stage ----------
FROM python:3.11-slim AS builder

WORKDIR /app

RUN pip install --upgrade pip

COPY requirements.txt .
RUN pip install --prefix=/install -r requirements.txt

COPY . .
RUN pip install --prefix=/install -e .

# ---------- Runtime Stage ----------
FROM python:3.11-slim

WORKDIR /app

COPY --from=builder /install /usr/local
COPY src/ /app/src/

ENV PYTHONPATH=/app/src

EXPOSE 8000

CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
''').strip() + '\n')


(PROJECT / '.dockerignore').write_text(textwrap.dedent('''
.git
.venv
__pycache__/
*.pyc
.ipynb_checkpoints/
data/
.env
''').strip() + '\n')

61

### 3.3 Build and compare image sizes

In [ ]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

In [19]:
# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    '<!-- TODO: Write 2-3 sentences explaining why the sizes differ and which you would use in production. -->',
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')

          Repository       Tag   Size
qbc12-airbnb-serving optimized 1.29GB
qbc12-airbnb-serving     naive 3.15GB

Report saved to reports/docker_size_report.md


### 3.4 Docker Compose

In [20]:
# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

# Write your code here.

(PROJECT / 'docker-compose.yml').write_text(textwrap.dedent('''
services:
  airbnb-serving:
    image: qbc12-airbnb-serving:optimized
    container_name: airbnb-serving
    ports:
      - "8000:8000"
    env_file:
      - .env
    restart: unless-stopped
''').strip() + '\n')

(PROJECT / '.env').write_text(textwrap.dedent('''
MLFLOW_TRACKING_URI=http://185.50.38.163:33014
QBC12_STUDENT_ID = "student_elaheh_badali"
MLFLOW_TRACKING_USERNAME = "student_elaheh_badali"
MLFLOW_TRACKING_PASSWORD = "AB9XBnbNX4VW6YSgRkY"
MODEL_RUN_ID=6850fbb8dc784ea0850400865ddae4a1
''').strip() + '\n')


(PROJECT / '.gitignore').write_text(textwrap.dedent('''
.env
__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
data/
''').strip() + '\n')

57

In [21]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

 Container airbnb-serving  Running


Docker Compose health check passed: {'status': 'ok', 'model_run_id': '6850fbb8dc784ea0850400865ddae4a1'}


 Container airbnb-serving  Stopping
 Container airbnb-serving  Stopped
 Container airbnb-serving  Removing
 Container airbnb-serving  Removed
 Network 01_default  Removing
 Network 01_default  Removed


---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [25]:
# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

# Write your code here.

(PROJECT / 'k8s').mkdir(exist_ok=True)

(PROJECT / 'k8s/deployment.yaml').write_text(textwrap.dedent('''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
        - name: airbnb-serving
          image: qbc12-airbnb-serving:optimized
          ports:
            - containerPort: 8000
          env:
            - name: MLFLOW_TRACKING_URI
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_URI
            - name: MLFLOW_TRACKING_USERNAME
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_USERNAME
            - name: MLFLOW_TRACKING_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MLFLOW_TRACKING_PASSWORD
            - name: MODEL_RUN_ID
              valueFrom:
                secretKeyRef:
                  name: airbnb-serving-secret
                  key: MODEL_RUN_ID
          resources:
            requests:
              cpu: 100m
              memory: 256Mi
            limits:
              cpu: 500m
              memory: 512Mi
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 15
            periodSeconds: 10
''').strip() + '\n')

1461

### 4.2 Service

In [26]:
# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment

# Write your code here.
(PROJECT / 'k8s').mkdir(exist_ok=True)

(PROJECT / 'k8s/service.yaml').write_text(textwrap.dedent('''
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
    - port: 80
      targetPort: 8000
''').strip() + '\n')

169

### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With 1 replica traffic stops if the Pod crashes, with 2 replicas traffic is automatically routed to the healthy Pod.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: We need delay because the model loads from MLflow and FastAPI takes time before it can respond to /health.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Secrets keep credentials secure and separate from code so they are not exposed in YAML or version control.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: ClusterIP is internal-only access inside the cluster, while LoadBalancer exposes the service to external internet traffic.

---
## Final Proof

If this cell fails, HW03 is not complete.

In [27]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```